# 🚀 VoiceCash NLP Benchmark: Llama 3.1 8B Instruct (4-bit GPU)

Notebook này được thiết kế để chạy benchmark mô hình **Meta-Llama/Llama-3.1-8B-Instruct** (hoặc Groq baseline) cho hệ thống kế toán giọng nói **VoiceCash** trên **Google Colab GPU** (T4 / A100 / L4).

### 🎯 Mục tiêu:
1. Chạy hoàn toàn độc lập, **KHÔNG kết nối database/backend production**.
2. Sử dụng **Hugging Face Transformers** với **4-bit NF4 Quantization (BitsAndBytes)** để tối ưu VRAM.
3. Streaming persistence vào file `.jsonl`, tự động **resume** nếu Colab bị gián đoạn.
4. Đánh giá offline đầy đủ (Accuracy, Macro/Weighted F1, Confusion Matrix, Per-entity metrics, Strict vs Normalized E2E match).
5. Xuất đầy đủ các báo cáo JSON, CSV và Markdown.

## 1. Kiểm tra phần cứng GPU & CUDA

In [ ]:
!nvidia-smi

import torch
print("=" * 50)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Name       : {gpu_name}")
    print(f"Total VRAM     : {total_vram:.2f} GB")
else:
    print("[!] CẢNH BÁO: Chưa bật GPU! Hãy vào Menu: Runtime -> Change runtime type -> T4 GPU")
print("=" * 50)

## 2. Mount Google Drive (Tùy chọn) & Chuẩn bị thư mục làm việc

In [ ]:
# 1. Nếu bạn tải file ai_benchmark.zip lên Colab, tự động giải nén:
import os
if os.path.exists("ai_benchmark.zip"):
    !unzip -q -o ai_benchmark.zip
    print("[✓] Đã giải nén ai_benchmark.zip thành công!")

# 2. Đảm bảo thư mục làm việc nằm trong sys.path
import sys
cwd = os.getcwd()
if cwd not in sys.path:
    sys.path.insert(0, cwd)
print(f"Current working directory: {cwd}")

## 3. Cài đặt Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q transformers bitsandbytes accelerate scikit-learn pandas tqdm pydantic python-dotenv huggingface_hub
print("[✓] Đã cài đặt xong dependencies!")

## 4. Xác thực Hugging Face Token (Bảo mật - KHÔNG hard-code)

> **Lưu ý**: Llama 3.1 8B là gated model của Meta. Bạn cần chấp thuận điều khoản tại [meta-llama/Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct).
> Lưu `HF_TOKEN` trong mục **Secrets** (biểu tượng chiếc chìa khóa bên trái Colab) hoặc nhập an toàn vào ô bên dưới.

In [ ]:
import os
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print("[✓] Đã lấy HF_TOKEN từ Colab Secrets!")
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if not hf_token:
    from getpass import getpass
    print("Vui lòng nhập Hugging Face Token (Read access):")
    hf_token = getpass()

login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
print("[✓] Đăng nhập Hugging Face thành công!")

## 5. Khởi tạo mô hình Llama 3.1 8B Instruct (4-bit NF4)

In [ ]:
from ai_benchmark.models.llama_local import LlamaLocalModel

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

print("[*] Bắt đầu load Llama 3.1 8B Instruct 4-bit NF4...")
model = LlamaLocalModel(
    model_name=MODEL_NAME,
    load_in_4bit=True
)

hw = model.get_hardware_info()
print("=" * 50)
print(f"[✓] Model Loaded Successfully: {MODEL_NAME}")
print(f"    Device     : {hw['device']}")
print(f"    GPU Name   : {hw['gpu_name']}")
print(f"    VRAM Đã cấp: {hw['vram_used_mb']} MB / {hw['vram_total_mb']} MB")
print("=" * 50)

## 6. Xem trước Tập Dữ Liệu (Dataset Inspection)

In [ ]:
import pandas as pd
from ai_benchmark.config import BenchmarkConfig

df = pd.read_csv(BenchmarkConfig.DEFAULT_DATASET_PATH)
print(f"Tổng số mẫu trong dataset: {len(df)}")
print("\nPhân bố Intent:")
print(df['intent'].value_counts())
display(df.head(5))

## 7. Chạy Benchmark Inference (Streaming & Resume Support)

- Chạy thử nghiệm trên 100–200 mẫu (có thể chỉnh biến `TEST_SAMPLES`).
- Mỗi kết quả được ghi ngay lập tức vào file `.jsonl`.
- Nếu notebook bị ngắt kết nối giữa chừng, chỉ cần chạy lại cell này, hệ thống sẽ tự động bỏ qua các mẫu đã hoàn thành.

In [ ]:
from ai_benchmark.inference import BenchmarkPredictor

TEST_SAMPLES = 200  # Số mẫu benchmark (100-200 mẫu)

predictor = BenchmarkPredictor(
    model_provider=model,
    dataset_path=BenchmarkConfig.DEFAULT_DATASET_PATH
)

summary = predictor.run(
    limit=TEST_SAMPLES,
    stratified=False
)

## 8. Chạy Đánh Giá Offline (Offline Evaluation)

- Đánh giá Intent Classification (Accuracy, Macro & Weighted F1).
- Đánh giá Slot Filling (Precision, Recall, F1 cho 9 entities).
- Đánh giá End-to-End Strict Match & Normalized Match.
- Tạo Confusion Matrix.

In [ ]:
from ai_benchmark.evaluation import BenchmarkEvaluator

evaluator = BenchmarkEvaluator(
    dataset_path=BenchmarkConfig.DEFAULT_DATASET_PATH,
    output_dir=BenchmarkConfig.OUTPUT_DIR
)

eval_results = evaluator.evaluate(
    predictions_path=predictor.predictions_file
)

evaluator.print_summary(eval_results)

## 9. Hiển thị Báo Cáo Chi Tiết và Confusion Matrix

In [ ]:
import os
from IPython.display import Markdown, display

report_md_path = BenchmarkConfig.OUTPUT_DIR / "evaluation_report.md"
if report_md_path.exists():
    with open(report_md_path, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))

print("\nClassification Report per Intent:")
display(pd.read_csv(BenchmarkConfig.OUTPUT_DIR / "classification_report.csv"))

print("\nEntity Extraction Metrics per Entity:")
display(pd.read_csv(BenchmarkConfig.OUTPUT_DIR / "entity_metrics.csv"))

print("\nConfusion Matrix:")
display(pd.read_csv(BenchmarkConfig.OUTPUT_DIR / "confusion_matrix.csv", index_col=0))

## 10. Tải về gói kết quả đánh giá (Reports + Predictions + Summary)

In [ ]:
!zip -r voicecash_llama31_benchmark_results.zip ai_benchmark/evaluation/reports ai_benchmark/predictions

try:
    from google.colab import files
    files.download('voicecash_llama31_benchmark_results.zip')
    print("[✓] Đang tải file kết quả về máy của bạn...")
except Exception:
    print("File đã được tạo: voicecash_llama31_benchmark_results.zip")